# 5.5 · B-trees and block storage

*Trees* · 9 exercises

Count block reads instead of steps, and pack many keys per node to keep the tree shallow.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-5.5) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-5.5) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-5.5)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import bisect
import math
import random

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 5.5.1 · Count I/Os, not steps  `Core`

**Task.** Memory moves in blocks of B items. Write `scan_ios(n, B)` for reading n items in order, and `random_reads_ios(k)` for k reads that each land in a different block.

**Example.** `scan_ios(1000, 64)` -> `16`; `random_reads_ios(1000)` -> `1000`

**Big O.** A scan costs O(N / B) I/Os; scattered access costs one I/O per item. On disk, I/Os dominate the running time.

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-5-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-5-1)

In [ ]:
# 5.5.1 Count I/Os, not steps
def scan_ios(n, B):
    raise NotImplementedError


def random_reads_ios(k):
    raise NotImplementedError

### 5.5.2 · Binary search over blocks  `Core`

**Task.** Write `ExternalArray(data, B)` whose `read_block(i)` returns block i and counts one I/O, and `external_binary_search(ext, target)` that binary-searches the blocks by their first item, then searches inside one block in memory.

**Example.** 10,000 sorted items with B = 100 -> a lookup costs about 8 I/Os, versus about 14 for plain binary search touching one block per probe

**Big O.** O(log(N / B)) I/Os, a small saving; the B-tree (5.5.4) does much better with O(log_B N).

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-5-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-5-2)

In [ ]:
# 5.5.2 Binary search over blocks
class ExternalArray:
    def __init__(self, data, B):
        raise NotImplementedError

    def read_block(self, i):
        raise NotImplementedError


def external_binary_search(ext, target):
    raise NotImplementedError

### 5.5.3 · Lookups: BST versus B-tree  `Core`

**Task.** Write `bst_ios(n)` (one I/O per level of a balanced binary tree, log2 N) and `btree_ios(n, B)` (log base B of N) and compare them for a billion keys.

**Example.** `bst_ios(10**9)` -> `30`; `btree_ios(10**9, 1000)` -> `3`

**Big O.** O(log_2 N) versus O(log_B N) I/Os: a factor of log2 B fewer.

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-5-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-5-3)

In [ ]:
# 5.5.3 Lookups: BST versus B-tree
def bst_ios(n):
    raise NotImplementedError


def btree_ios(n, B):
    raise NotImplementedError

### 5.5.4 · B-tree nodes and search  `Core`

**Task.** Create `BTreeNode(leaf)` with sorted `keys` and `children`, and `BTree(t)` with a `search(key)` that binary-searches each node's keys and descends into the right child.

**Example.** in a B-tree holding 1..100 -> `search(42)` is `True`, `search(101)` is `False`

**Big O.** O(log_t N) nodes visited, O(log t) comparisons in each.

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-5-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-5-4)

In [ ]:
# 5.5.4 B-tree nodes and search
class BTreeNode:
    def __init__(self, leaf=True):
        raise NotImplementedError


class BTree:
    def __init__(self, t=3):
        raise NotImplementedError

    def search(self, key, node=None):
        raise NotImplementedError

### 5.5.5 · Insert with proactive splits  `Core`

**Task.** Add `insert(key)`: if the root is full (2t - 1 keys), split it and grow a new root. Walk down, splitting any full child before entering it, so a leaf always has room.

**Example.** t = 2, insert 1..10 -> `validate()` passes and every leaf sits at the same depth

**Big O.** O(t log_t N) CPU, O(log_t N) node reads. The tree grows only at the root, which keeps it perfectly balanced.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-5-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-5-5)

In [ ]:
# 5.5.5 Insert with proactive splits
def split_child(self, parent, i):
    raise NotImplementedError

def insert(self, key):
    raise NotImplementedError


BTree.split_child = split_child
BTree.insert = insert

In [ ]:
# Test helpers: run once.
def btree(t, keys):
    tree = BTree(t)
    for k in keys:
        tree.insert(k)
    return tree

In [ ]:
def test_ex_5_5_1():
    assert scan_ios(1000, 64) == 16
    assert random_reads_ios(1000) == 1000


test_ex_5_5_1()
print('5.5.1 passed')

In [ ]:
def test_ex_5_5_2():
    ext = ExternalArray(range(0, 20000, 2), 100)
    assert external_binary_search(ext, 4242) == 2121
    assert ext.ios <= 9
    assert external_binary_search(ExternalArray(range(0, 20000, 2), 100), 4243) == -1
    assert external_binary_search(ExternalArray(range(10), 3), 0) == 0
    assert external_binary_search(ExternalArray(range(10), 3), 9) == 9


test_ex_5_5_2()
print('5.5.2 passed')

In [ ]:
def test_ex_5_5_3():
    assert bst_ios(10 ** 9) == 30
    assert btree_ios(10 ** 9, 1000) == 3


test_ex_5_5_3()
print('5.5.3 passed')

In [ ]:
def test_ex_5_5_4():
    t = BTree(2)
    assert not t.search(1)
    left, right = (BTreeNode(leaf=True), BTreeNode(leaf=True))
    left.keys, right.keys = ([1, 2], [4, 5])
    t.root = BTreeNode(leaf=False)
    t.root.keys, t.root.children = ([3], [left, right])
    assert all((t.search(k) for k in (1, 2, 3, 4, 5)))
    assert not t.search(0)
    assert not t.search(6)


test_ex_5_5_4()
print('5.5.4 passed')

In [ ]:
def test_ex_5_5_5():
    t = btree(2, range(1, 11))
    assert t.search(7)
    assert not t.search(11)
    assert all((t.search(k) for k in range(1, 11)))
    assert all((len(n.keys) <= 3 for n in [t.root] + t.root.children))


test_ex_5_5_5()
print('5.5.5 passed')

### 5.5.6 · Validate and walk in order  `Core`

**Task.** Add `inorder()` returning all keys sorted, and `validate()` that checks key counts per node, sorted keys, child counts, and that all leaves share one depth. Return the height.

**Example.** after random inserts and deletes -> `inorder()` equals the sorted set of keys and `validate()` returns the height

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-5-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-5-6)

In [ ]:
# 5.5.6 Validate and walk in order
def inorder(self, node=None):
    raise NotImplementedError

def validate(self):
    raise NotImplementedError


BTree.inorder = inorder
BTree.validate = validate

In [ ]:
def test_ex_5_5_6():
    t = BTree(2)
    for k in range(1, 11):
        t.insert(k)
        t.validate()
    assert t.inorder() == list(range(1, 11))


test_ex_5_5_6()
print('5.5.6 passed')

### 5.5.8 · Range queries, like a database index  `Practice`

**Task.** Add `range_query(lo, hi)` returning the keys in [lo, hi] in order, skipping whole subtrees that cannot contain them.

**Example.** B-tree of 1..100 -> `range_query(40, 45)` is `[40, 41, 42, 43, 44, 45]`

**Big O.** O(log_t N + K / t) node visits for K results. This is why databases index with B-trees.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-5-8) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-5-8)

In [ ]:
# 5.5.8 Range queries, like a database index
def range_query(self, lo, hi, node=None, out=None):
    raise NotImplementedError


BTree.range_query = range_query

In [ ]:
def test_ex_5_5_8():
    t = btree(3, range(1, 101))
    assert t.range_query(40, 45) == [40, 41, 42, 43, 44, 45]
    assert t.range_query(0, 3) == [1, 2, 3]
    assert t.range_query(200, 300) == []


test_ex_5_5_8()
print('5.5.8 passed')

### 5.5.9 · Deletion  `Stretch`

**Task.** Add `delete(key)`. Before descending into a child with only t - 1 keys, top it up by borrowing from a sibling or merging with one. Delete from a leaf directly; for an internal key, replace it with its predecessor or successor, or merge. Shrink the root when it empties.

**Example.** t = 3, insert 1..200, delete the odd keys in shuffled order -> `validate()` passes and `inorder()` is the even keys

**Big O.** O(log_t N) node visits.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-5-9) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-5-9)

In [ ]:
# 5.5.9 Deletion
def delete(self, key):
    raise NotImplementedError

def _delete(self, node, key):
    raise NotImplementedError

def _merge(self, node, i):
    raise NotImplementedError


BTree.delete = delete
BTree._delete = _delete
BTree._merge = _merge

In [ ]:
def test_ex_5_5_9():
    rng = random.Random(8)
    for t_deg in (2, 3, 4):
        t, keys = (BTree(t_deg), set())
        for _ in range(1500):
            k = rng.randrange(400)
            if rng.random() < 0.55:
                t.insert(k)
                keys.add(k)
            else:
                assert t.delete(k) == (k in keys)
                keys.discard(k)
        t.validate()
        assert t.inorder() == sorted(keys)
    t = btree(3, range(1, 201))
    odds = list(range(1, 201, 2))
    random.Random(3).shuffle(odds)
    for k in odds:
        t.delete(k)
    t.validate()
    assert t.inorder() == list(range(2, 201, 2))


test_ex_5_5_9()
print('5.5.9 passed')

### 5.5.7 · How tall can a B-tree get?  `Core`

**Task.** Write `btree_height_bound(n, t)` returning the most levels a B-tree of minimum degree t with n keys can have: 1 + floor(log_t((n + 1) / 2)). Check it against real trees.

**Example.** `btree_height_bound(10**6, 50)` -> `4`; a real tree of 1..10,000 with t = 3 stays within its bound of 8

**Big O.** O(1). With t in the hundreds, a billion keys fit in 4 or 5 levels.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-5-7) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-5-7)

In [ ]:
# 5.5.7 How tall can a B-tree get?
def btree_height_bound(n, t):
    raise NotImplementedError

In [ ]:
def test_ex_5_5_7():
    assert btree_height_bound(10 ** 6, 50) == 4
    assert btree(3, range(1, 10001)).validate() <= btree_height_bound(10000, 3)


test_ex_5_5_7()
print('5.5.7 passed')